# 🔌 3. MCP & Skills: 데이터 소스를 서버로 분리하고, 에이전트가 스스로 쓰게 하기

1교시의 에이전트는 검색 도구를 **에이전트 코드 안에 직접** 바인딩해서 사용했습니다.
하지만 엔터프라이즈 환경에서 데이터 소스가 수십 개로 늘어나고 각 데이터베이스의 소유권(Governance)과 배포 주기가 나뉘면, 도구를 에이전트 코드에 직접 하드코딩하는 방식은 즉각 한계에 직면합니다.

이번 시간에는 데이터 접근 계층을 독립적인 마이크로서비스인 **MCP (Model Context Protocol) 서버**로 분리하고,
에이전트가 **Skills 패턴(점진적 공개: Progressive Disclosure)** 을 통해 필요한 서버와 도구를 자율적으로 탐색하여 실행하는 차세대 에이전틱 아키텍처를 구축합니다.

---

### 🎯 학습 목표
1. **FastMCP 프레임워크**로 MCP 표준 서버를 구축하고, Pydantic 스키마를 통해 LLM이 생성한 비정상 인자를 서버 레벨에서 엄격히 차단합니다.
2. **Bearer 토큰 인증과 Scope 인가**를 적용하여 허가된 에이전트만 서버를 사용하게 통제하고, 민감한 인증 토큰이 프롬프트나 로그에 영구 노출되지 않는 보안 패턴을 구현합니다.
3. 동일한 MCP 서버를 **정적 바인딩(`MultiServerMCPClient`)** 방식과 **Skills 기반 점진적 공개(Progressive Disclosure)** 방식으로 각각 조립해 보고, 토큰 소비량과 시스템 확장성 관점에서 두 아키텍처의 트레이드오프를 비교 분석합니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 프로젝트 루트 탐색, `.env`, `nest_asyncio` |
| **Part 1** | 왜 MCP인가 (아키텍처 배경) | 데이터 접근 계층 분리, MCP 3대 Primitives, 도구 폭발 문제 |
| **Part 2** | FastMCP 서버 구축 & 스키마 검증 | 파이썬 데코레이터(`@mcp.tool`), Pydantic 검증, In-Process 테스트 |
| **Part 3** | 엔터프라이즈 인증/인가 (Auth) | Bearer Token, `StaticTokenVerifier`, `require_scopes`, 401 차단 |
| **Part 4** | 연결 방식 A: 정적 바인딩 | `langchain_mcp_adapters`로 일괄 로드 및 한계(Context Bloat) |
| **Part 5** | 연결 방식 B: Skills 기반 점진적 공개 | 범용 도구 3종, 스킬 카탈로그 주입, `--token-env` 보안, 자율 탐색 |
| **Part 6** | 정리와 Mission 03 | 프로덕션 서버 구현 및 두 에이전트 대화 비교 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)

print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 공통 유틸리티 로드
from app.utils import init_chat_model, normalize_content

import logging
logging.getLogger("fastmcp").setLevel(logging.CRITICAL)                 # 서버 측 검증 오류 로그는 출력하지 않음
logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)     # 스키마 변환 경고 생략

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)

---
## 🧭 Part 1. 왜 MCP인가: 데이터 접근 계층의 분리와 개방형 표준

### 1.1 도구 직접 바인딩의 한계 vs MCP 마이크로서비스 아키텍처

1교시의 에이전트는 데이터베이스 접속 객체와 검색 로직을 에이전트 프로세스 내부에 직접 품고 있었습니다. 이는 프로토타입 단계에서는 편리하지만, 실무 환경에서는 다음과 같은 심각한 병목을 만듭니다:

| 비교 항목 | 도구를 에이전트 코드에 직접 포함 (Monolithic) | MCP 서버로 분리 (Microservice) |
|:---|:---|:---|
| **소유와 배포 주기** | 데이터 엔지니어가 검색 로직을 수정할 때마다 **에이전트 전체를 재배포**해야 함 | **데이터 팀이 MCP 서버만 독립 배포**. 에이전트는 무중단 서비스 유지 |
| **권한 및 보안** | 에이전트 실행 런타임이 모든 DB 접속 키와 원천 데이터를 직접 보유함 | **MCP 서버가 인증·인가 게이트웨이 역할** 수행. 에이전트는 토큰만 소유 |
| **크로스 플랫폼 재사용** | 에이전트마다 동일한 DB 검색 도구 코드를 중복 구현/복사해야 함 | 사내 다양한 에이전트, 웹 서비스, Claude Desktop 등이 **단일 서버를 표준 프로토콜로 공유** |
| **언어 및 런타임 격리** | 에이전트 런타임(Python)과 동일한 환경으로 도구 구현이 제한됨 | MCP 서버는 Node.js, Go, Rust 등 **데이터 소스에 최적화된 언어로 구축 가능** |

---

### 1.2 Model Context Protocol (MCP)이란?

**MCP(Model Context Protocol)** 는 2024년 11월 Anthropic이 오픈소스로 공개한 **AI 애플리케이션과 외부 데이터 소스/도구 간의 범용 개방형 표준 프로토콜**입니다. 하드웨어의 USB-C 포트처럼, 모델과 데이터 간의 표준화된 규격을 정의합니다.

#### 📌 MCP의 3대 핵심 Primitives (기본 구성요소)
1. **Tools (도구)**:
   - 모델이 계산, 외부 API 호출, DB 쿼리 등 **부수 효과(Side-effect)**를 일으키며 실행할 수 있는 원격 RPC 함수입니다.
   - Pydantic 또는 JSON Schema를 통해 매개변수 규격을 엄격히 정의합니다.
2. **Resources (리소스)**:
   - 파일 내용, 데이터베이스 테이블 스키마, 시스템 로그 등 **읽기 전용(Read-only) 데이터 컨텍스트**를 제공하는 파일/URI 인터페이스입니다. (GET 요청과 유사)
3. **Prompts (프롬프트)**:
   - 서버가 클라이언트에게 제공하는 **사전 정의된 슬래시 커맨드 및 재사용 가능한 프롬프트 템플릿**입니다.

#### 🌐 전송 방식 (Transports)의 진화: SSE에서 Stateless Streamable HTTP로
- **`stdio` (표준 입출력)**: 로컬 CLI 환경이나 데스크톱 앱(Claude Desktop 등)에서 하위 프로세스를 직접 띄워 표준 입출력 파이프로 통신할 때 사용합니다.
- **`Streamable HTTP` (HTTP 방식)**: 최신 MCP 사양과 FastMCP는 이를 단일 엔드포인트(예: `/mcp`)에 표준 HTTP POST를 보내는 **Stateless(무상태) Streamable HTTP**로 요청마다 세션 없이 독립적으로 처리되므로 로드밸런서 뒤에서 자유롭게 수평 확장(Auto-scaling)할 수 있으며, 응답은 표준 스트리밍 형식으로 안전하게 전달됩니다.

---

### 1.3 새로운 과제: 도구 폭발 (Tool Explosion)
MCP로 데이터 소스를 마이크로서비스화하면 접근성이 비약적으로 높아지지만, 서버가 10개, 도구가 100개로 늘어나면 **"100개의 도구 스키마를 어떻게 에이전트에게 전달할 것인가?"**라는 새로운 아키텍처 문제가 발생합니다.
이 문제는 Part 4(정적 바인딩)와 Part 5(Skills 점진적 공개)에서 심도 있게 비교합니다.

---
## 🛠️ Part 2. FastMCP 서버 구축 & Pydantic 스키마 검증

**FastMCP**는 FastAPI 스타일의 직관적인 데코레이터(`@mcp.tool`, `@mcp.resource`)를 사용하여 수십 줄의 보일러플레이트 코드 없이 파이썬 함수를 즉시 MCP 표준 서버로 만들어 주는 고수준 프레임워크입니다.

먼저 네트워크 포트 바인딩 없이, **같은 Python 프로세스 메모리 내(In-Process)**에서 서버와 클라이언트를 연결하여 동작 원리와 스키마 검증 메커니즘을 확인합니다.

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field
from fastmcp import FastMCP, Client
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

logging.getLogger("fastmcp").setLevel(logging.CRITICAL)   # fastmcp는 import 시 로그 레벨을 다시 설정하므로 import 뒤에 지정

policy_store = Chroma(
    collection_name="enterprise_policy_store",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-large"),
    persist_directory="app/database/chroma_db",
)

demo_server = FastMCP(name="policy-demo", instructions="사내 규정을 검색하는 데모 서버")


@demo_server.tool(description="사내 규정을 검색하여 관련 조항을 반환합니다.")
def search_policy(query: str) -> str:
    docs = policy_store.similarity_search(query, k=1)
    return docs[0].metadata["breadcrumb"] + "\n" + docs[0].metadata["raw_content"] if docs else "결과 없음"


# Pydantic 스키마: LLM이 잘못된 인자를 보내면 서버에서 거부됩니다.
class Grade(str, Enum):
    EXECUTIVE = "임원"
    LEADER = "팀장"
    MEMBER = "팀원"


class AllowanceQuery(BaseModel):
    grade: Grade = Field(description="직급 구분")
    days: int = Field(ge=1, le=30, description="출장 일수 (1~30)")


@demo_server.tool(description="직급과 출장 일수로 일비 총액을 계산합니다. (제15조 기준)")
def calc_daily_allowance(query: AllowanceQuery) -> str:
    per_day = {"임원": 400000, "팀장": 300000, "팀원": 250000}[query.grade.value]
    return f"{query.grade.value} {query.days}일 일비: {per_day * query.days:,}원"

In [ ]:
async def demo_in_process1():
    async with Client(demo_server) as client:
        tools = await client.list_tools()
        print("📋 도구 목록:", [t.name for t in tools])

await demo_in_process1()

In [ ]:
async def demo_in_process2():
    async with Client(demo_server) as client:
        try:
            r = await client.call_tool("calc_daily_allowance", {"query": {"grade": "팀장", "days": 3}})
            print("✅ 정상 호출:", r.data)
        except Exception as e:
            print("❌ 스키마 위반 차단:", type(e).__name__)

        try:
            r = await client.call_tool("calc_daily_allowance", {"query": {"grade": "과장", "days": 99}})
            print("✅ 정상 호출:", r.data)
        except Exception as e:
            print("❌ 스키마 위반 차단:", type(e).__name__)

await demo_in_process2()

In [ ]:
async def demo_in_process3():
    async with Client(demo_server) as client:
        r = await client.call_tool("search_policy", {"query": "정산 신청 기한"})
        print("\n✅ 규정 검색:\n", r.data)

await demo_in_process3()

> ### 🛡️ LLM과 MCP 백엔드 사이의 방어선
> LLM은 확률적 모델이므로, 사내 규정에 존재하지 않는 직급(`"과장"`)이나 물리적으로 불가능한 값(`99일`)을 도구 인자로 보낼 수 있습니다.
> 
> 1. FastMCP 서버는 MCP 도구 입력의 표준 **JSON Schema**를 Pydantic 모델과 파이썬 타입 힌트를 통해 자동 생성하고, 이를 LLM에게 도구 스펙으로 제공합니다.
> 2. LLM이 호출을 시도할 때, FastMCP는 비즈니스 로직을 실행하기 전에 **Pydantic 유효성 검증을 먼저 수행(Fail-fast)**합니다.
> 3. 규격을 위반한 요청은 즉시 차단되어 DB 과부하나 쿼리 에러를 미연에 방지합니다.
>
> 또한 `Client(demo_server)`와 같은 **In-Process 클라이언트**는 복잡한 포트 바인딩 없이도 단위 테스트(Unit Test)를 빠르게 수행할 수 있는 강력한 도구입니다.

---
## 🔐 Part 3. 엔터프라이즈 인증/인가 (Auth) 아키텍처

사내 규정, 재무 지표, 프로젝트 예산 등 기업 데이터는 모든 에이전트에게 무제한으로 열려 있어서는 안 됩니다. 데이터 접근 계층을 마이크로서비스로 분리했다면, **반드시 접근 제어(Access Control)가 동반**되어야 합니다.

실제 프로덕션용 서버인 `app/mcp/enterprise_rag_server.py`는 **FastMCP의 2계층 보안(Bearer Token 인증 + Scope 기반 인가)** 을 적용하고 있습니다. 인증 관련 코드만 확인합니다.

In [ ]:
src = open("app/mcp/enterprise_rag_server.py", encoding="utf-8").read()
start = src.find("# 1. 인증기")
print(src[start:src.find("DB_DIR =")])
print("...")
tool_start = src.find("@mcp.tool(")
print(src[tool_start:src.find("def search_company_policy")])

### 🛡️ 2계층 보안 아키텍처 (인증 vs 인가)와 FastMCP의 동작 원리

| 보안 계층 | 질문 | FastMCP 서버 구현 코드 | 클라이언트 전송 방식 | 실패 시 동작 |
|:---|:---|:---|:---|:---|
| **1계층: 인증 (Authentication)** | *"너 신원 확인된 에이전트 맞아?"* | `FastMCP(..., auth=verifier)` | HTTP Header: `Authorization: Bearer <토큰>` | 토큰 누락 또는 불일치 시 `401 Unauthorized` 연결 차단 |
| **2계층: 인가 (Authorization)** | *"접속은 됐는데, 이 도구 쓸 권한 있어?"* | `@mcp.tool(auth=require_scopes("rag:read"))` | 인증 시 획득한 `scopes`로 자동 검증 | 토큰에 `rag:read` 권한이 없으면 도구 실행 거부 (`403 Forbidden`) |

---

#### 1. 서버 측 인증/인가 구축 원리 (`enterprise_rag_server.py`)

서버 코드는 다음 3단계로 보안을 강제합니다:
1. **토큰-권한 매핑 (`StaticTokenVerifier`)**:
   `verifier = StaticTokenVerifier(tokens={SERVER_TOKEN: {"client_id": "...", "scopes": ["rag:read"]}})`
   서버는 유효한 토큰 목록과, 각 토큰이 보유한 권한(`scopes`)을 화이트리스트 사전으로 관리합니다.
2. **서버 전역 인증 게이트 (`FastMCP(..., auth=verifier)`)**:
   FastMCP 내부 미들웨어가 모든 들어오는 HTTP 요청의 헤더를 가로채 검사합니다.
   유효한 토큰이 확인되면 클라이언트의 권한 정보(`scopes`)를 요청 컨텍스트(`AuthContext`)에 안전하게 저장합니다.
3. **도구별 실행 인가 (`require_scopes`)**:
   `@mcp.tool(auth=require_scopes("rag:read"))` 데코레이터는 도구를 실행하기 직전, 토큰에 `"rag:read"` 스코프가 실제로 들어있는지 대조하여 무권한 실행을 방지합니다.

---

이제 백그라운드 프로세스로 실제 MCP 서버(`app/mcp/enterprise_rag_server.py`)를 포트 8010에 기동하고,
클라이언트의 3가지 인증 시나리오를 직접 검증해 봅니다.

In [ ]:
import subprocess
import time
import httpx

MCP_URL = "http://localhost:8010/mcp"

def is_server_up(url: str) -> bool:
    try:
        httpx.get(url, timeout=2)   # 인증 서버이므로 401이어도 '떠 있음'
        return True
    except httpx.HTTPError:
        return False

mcp_process = None
if is_server_up(MCP_URL):
    print("ℹ️ 이미 실행 중인 MCP 서버를 사용합니다:", MCP_URL)
else:
    os.makedirs("artifacts", exist_ok=True)
    mcp_process = subprocess.Popen(
        [sys.executable, "app/mcp/enterprise_rag_server.py", "--port", "8010"],
        stdout=open("artifacts/mcp_server.log", "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if is_server_up(MCP_URL):
            break
        time.sleep(1)
    print(f"🚀 MCP 서버 기동 완료 (PID {mcp_process.pid}) → {MCP_URL}")

In [ ]:
from fastmcp.client.auth import BearerAuth

TOKEN = os.environ["ENTERPRISE_RAG_MCP_TOKEN"]   # 값은 출력하지 않습니다

async def try_connect(label, auth):
    try:
        async with Client(MCP_URL, auth=auth) as client:
            tools = await client.list_tools()
            print(f"✅ {label}: 접속 성공, 도구 {len(tools)}개 → {[t.name for t in tools]}")
    except Exception as e:
        print(f"❌ {label}: {str(e).splitlines()[0][:80]}")

await try_connect("토큰 없음", None)
await try_connect("틀린 토큰", BearerAuth("wrong-token"))
await try_connect("올바른 토큰", BearerAuth(TOKEN))

#### 2. 클라이언트의 `auth` 파라미터와 `BearerAuth`

`async with Client(MCP_URL, auth=auth) as client:` 코드에서 **`auth`에는 무엇이 들어가는 것일까요?**

FastMCP의 `BearerAuth`는 내부적으로 HTTP 요청 헤더에 `Authorization: Bearer <토큰>`을 주입하는 역할을 합니다:
```python
# fastmcp 내부 BearerAuth 실제 구현 (5줄)
class BearerAuth(httpx.Auth):
    def __init__(self, token: str):
        self.token = SecretStr(token)

    def auth_flow(self, request):
        # 👈 HTTP 요청 헤더에 Authorization을 실어 보냅니다!
        request.headers["Authorization"] = f"Bearer {self.token.get_secret_value()}"
        yield request
```

따라서 아래 실습 코드의 3가지 테스트는 각각 다음과 같은 HTTP 요청을 보냅니다:
- **`auth=None`**: `Authorization` 헤더가 없음 ➜ 서버가 토큰 부재로 **`401 Unauthorized`** 차단
- **`auth=BearerAuth("wrong-token")`**: `Authorization: Bearer wrong-token` ➜ 서버 화이트리스트에 없는 토큰이므로 **`401 Unauthorized`** 차단
- **`auth=BearerAuth(TOKEN)`**: `.env`에 정의된 정규 토큰 전송 ➜ 서버 검증 통과(`200 OK`) 및 `rag:read` 스코프 획득 ➜ **도구 목록 정상 반환**

> [!NOTE]
> **실무 환경(Production)으로의 확장:**
> 본 실습에서는 고정 토큰을 검증하는 `StaticTokenVerifier`를 사용합니다.
> 실제 엔터프라이즈 환경에서는 사내 통합 인증 시스템(Keycloak, Azure AD, Okta 등)이 발급한 비대칭 암호화 키 기반의 **`JWTVerifier`** 로 교체하기만 하면 됩니다. FastMCP의 도구 데코레이터(`require_scopes`)와 클라이언트 호출 인터페이스는 100% 동일하게 유지됩니다.

---

---
## 🔗 Part 4. 연결 방식 A: 정적 바인딩 (`langchain_mcp_adapters`)

이제 구축된 MCP 서버를 에이전트에 연결하는 방법을 살펴봅니다.
가장 먼저 떠오르는 직관적인 방법은, 에이전트 시작 시점에 MCP 서버의 모든 도구를 긁어와(Fetch) LangChain 도구로 변환한 뒤 `create_agent`에 직접 주입하는 **정적 바인딩(Static Binding)** 방식입니다.

`langchain_mcp_adapters` 라이브러리의 `MultiServerMCPClient`를 사용하면 원격 MCP 서버의 HTTP 엔드포인트와 인증 헤더를 설정하여 도구들을 한 번에 로드할 수 있습니다.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

mcp_client = MultiServerMCPClient({
    "enterprise-rag": {
        "transport": "streamable_http",
        "url": MCP_URL,
        "headers": {"Authorization": f"Bearer {TOKEN}"},
    }
})
mcp_tools = await mcp_client.get_tools()
print("📦 정적 바인딩된 도구:", [t.name for t in mcp_tools])

langchain_mcp_adapters는 mcp 서버의 도구들을 langchain 도구로 변환해주는 어댑터입니다.

In [ ]:
mcp_tools

In [ ]:
static_agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="당신은 사내 데이터 Q&A 에이전트입니다. 도구 결과에 근거해서만 답하고 출처를 명시하세요.",
    checkpointer=InMemorySaver(),
)

async def ask(agent, question, thread_id):
    result = await agent.ainvoke(
        {"messages": [HumanMessage(content=question)]},
        {"configurable": {"thread_id": thread_id}},
    )
    last_human = max(i for i, m in enumerate(result["messages"]) if m.type == "human")
    calls = [tc for m in result["messages"][last_human:] for tc in (getattr(m, "tool_calls", None) or [])]
    for i, tc in enumerate(calls, 1):
        print(f"  🔄 Step {i}: {tc['name']}({str(tc['args'])[:160]})")
    print(f"  ➜ 도구 호출 {len(calls)}회\n")
    print("💬", normalize_content(result["messages"][-1].content)[:1200])
    return result

_ = await ask(static_agent, "정보보안 지침상 사외로 데이터를 반출할 때 승인 절차는?", "static-1")

### 💡 정적 바인딩의 한계: 도구 폭발 (Tool Explosion Problem)

정적 바인딩은 코드가 직관적이고 구현이 단순하지만, 데이터 소스가 늘어나는 순간 다음과 같은 심각한 한계에 부딪힙니다:

| 문제점 | 실무적 영향 및 원인 |
|:---|:---|
| **Context Bloat (토큰 낭비)** | 사내에 10개 MCP 서버가 있고 서버당 10개의 도구가 있다면, **100개 도구의 방대한 JSON Schema 전체가 매 대화 턴마다 시스템 프롬프트에 주입**됩니다. 질문 1개를 처리하는 데 기본 컨텍스트만 수만 토큰을 소모합니다. |
| **Tool Confusion (도구 환각 & 오선택)** | 프롬프트에 도구 설명서가 100개씩 나열되어 있으면, LLM의 주의 집중(Attention)이 분산되어 유사한 이름의 엉뚱한 도구를 호출하거나 엉뚱한 매개변수를 생성할 확률이 급격히 증가합니다. |
| **강한 결합과 재배포 비용 (Tight Coupling)** | 사내에 새로운 부서의 MCP 서버가 생기거나 기존 서버의 도구 스펙이 변경될 때마다, **에이전트 코드(`MultiServerMCPClient` 설정)를 직접 수정하고 에이전트 서비스를 재배포**해야 합니다. |

> [!WARNING]
> 즉, **"모든 도구를 미리 프롬프트에 쏟아붓는 방식"**은 엔터프라이즈 규모로 확장할 수 없습니다.
> 이에 대한 해법이 바로 Part 5의 **Anthropic Skills 패턴(점진적 공개)**입니다.

---
## 🧠 Part 5. 연결 방식 B: Anthropic Skills 기반 점진적 공개 (Progressive Disclosure)

Anthropic Agent Skills 패턴의 핵심 철학은 매우 단순합니다:
> *"에이전트에게 100개의 도구 스키마를 미리 쥐어주지 마라. 대신 **도구 사용 설명서(Skill 매뉴얼)**와 **3개의 범용 원시 도구(Primitive Tools)**만 주어라."*

Skills 방식에서 에이전트는 비즈니스 도구의 존재를 미리 알지 못합니다. 파일 시스템을 탐색하고 터미널 명령을 내릴 수 있는 기본 도구 3종(`glob_search`, `file_read`, `bash_command`)을 장착합니다.

```mermaid
flowchart TD
    subgraph Agent ["(create_agent)"]
        Tools["범용 원시 도구 (tools/)<br/>• glob_search<br/>• file_read<br/>• bash_command"]
    end

    subgraph LocalContext ["로컬 파일시스템"]
        Cat["1. 스킬 카탈로그 <br/> (매 턴 프롬프트 주입) <br/>• mcp: 원격 MCP 서버 CLI 연동 도구"]
        SkillMD["2. skills/mcp/SKILL.md<br/>• CLI 사용법 및 옵션 설명서"]
        MCPRegistry["3. skills/mcp/references/mcp_servers.json <br/>• 사내 MCP 서버 레지스트리"]
    end

    subgraph RemoteServer ["원격 마이크로서비스"]
        MCPServer["4. Enterprise-RAG-Server (:8010)<br/>• list_tools.py (도구 목록 조회)<br/>• execute_tool.py (도구 실행)"]
    end

    Agent -->|"1. 스킬 필요 판단"| Cat
    Agent -->|"2. file_read (매뉴얼 학습)"| SkillMD
    Agent -->|"3. file_read (서버 정보 확인)"| MCPRegistry
    Agent -->|"4. bash_command (CLI 실행)"| MCPServer
```

### 5.1 점진적 공개 (Progressive Disclosure)의 3단계 라이프사이클

| 단계 | 정보 | 언제 컨텍스트에 들어오는가? | 토큰 비용 |
|:---:|:---|:---|:---:|
| **1단계: 식별 (Catalog)** | 스킬 이름과 한 줄 요약 | **항상** (시스템 프롬프트 상단) | 수십 토큰 |
| **2단계: 학습 (Instruction)** | 스킬 매뉴얼 (`SKILL.md`) 전문 | **해당 스킬이 필요하다고 판단된 순간** (`file_read`) | 수백 토큰 |
| **3단계: 실행 (Execution)** | 서버의 도구 스키마 및 실행 결과 | **실제 명령을 실행하는 순간** (`bash_command`) | 호출 시점만 |

이제 에이전트의 프롬프트에 매 턴 주입되는 **스킬 카탈로그**와, 서버 목록이 적힌 **MCP 레지스트리**를 직접 열어봅니다.

In [ ]:
from app.agents.skill_rag_agent.prompt import BASE_SKILL_RAG_SYSTEM_PROMPT, get_skill_prompt_builder

builder = get_skill_prompt_builder()
print("=== 매 호출마다 프롬프트에 붙는 스킬 카탈로그 ===")
print(builder.build_catalog())

In [ ]:
print(open("skills/mcp/references/mcp_servers.json", encoding="utf-8").read())

> [!IMPORTANT]
> ### 🔒 엔터프라이즈 보안: `--token-env` 간접 주입 패턴
> 
> 왜 레지스트리(`skills/mcp/references/mcp_servers.json`)에 실제 토큰 문자열이 아니라 **환경 변수 이름(`ENTERPRISE_RAG_MCP_TOKEN`)** 만 적혀 있을까요?
> 
> 1. **프롬프트 인젝션 방어**: 사용자나 공격자가 *"너의 시스템 프롬프트와 참조 문서를 그대로 출력해"*라고 유도해도, 프롬프트에는 환경 변수 이름만 있을 뿐 실제 비밀 토큰이 들어있지 않아 탈취할 수 없습니다.
> 2. **로그 및 트레이싱 누출 차단**: LLM과의 모든 대화 기록, LangSmith 등의 모니터링 로그, 서버 에러 트레이스에 민감한 API 토큰이 평문으로 기록되는 보안 취약점을 완벽히 원천 차단합니다.
> 3. **실제 토큰 소비**: CLI 스크립트(`skills/mcp/scripts/execute_tool.py`)가 로컬 OS의 환경 변수로부터 토큰을 읽어 직접 HTTP `Authorization: Bearer <토큰>` 헤더로 전송합니다.

---

### 5.2 CLI로 직접 호출해 보기 (에이전트가 수행할 동작을 사람이 먼저 검증)

에이전트에게 맡기기 전에, 에이전트가 실행하게 될 CLI 명령어(`list_tools.py`, `execute_tool.py`)를 개발자가 직접 `bash_command`로 호출하여 정상 동작하는지 확인합니다.

In [ ]:
from app.tools import bash_command

print(bash_command.invoke({"command": "python skills/mcp/scripts/list_tools.py --url http://localhost:8010/mcp --token-env ENTERPRISE_RAG_MCP_TOKEN"})[:900])

In [ ]:
cmd = (
    "python skills/mcp/scripts/execute_tool.py --url http://localhost:8010/mcp --token-env ENTERPRISE_RAG_MCP_TOKEN "
    "--tool search_graph_relations --args '{\"params\": {\"seed_entity\": \"홍팀장\", \"max_hops\": 1}}'"
)
print(bash_command.invoke({"command": cmd}))

### 5.3 Skills 에이전트 조립 및 실행

이제 `app/agents/skill_rag_agent/`에 구현된 구조 그대로 에이전트를 조립합니다.
에이전트에게는 비즈니스 도구를 주지 않고, **범용 도구 3종(`glob_search`, `file_read`, `bash_command`)**과 매 턴 스킬 카탈로그를 프롬프트에 주입해 주는 **`SkillCatalogMiddleware`**만을 장착합니다.

복합 질문을 던지고, 에이전트가 스스로 **설명서 읽기 → 서버 레지스트리 확인 → 도구 목록 조회 → 도구 실행**의 점진적 탐색 루프를 거치는지 확인합니다.

In [ ]:
from app.tools import glob_search, file_read, bash_command
from app.middleware import SkillCatalogMiddleware

skill_agent = create_agent(
    model=llm,
    tools=[glob_search, file_read, bash_command],
    system_prompt=BASE_SKILL_RAG_SYSTEM_PROMPT,
    middleware=[SkillCatalogMiddleware(builder)],
    checkpointer=InMemorySaver(),
)

result = await ask(skill_agent, "AgenticAI팀이 총괄하는 프로젝트의 예산과 승인권자, 그리고 이 프로젝트가 의존하는 다른 프로젝트는?", "skill-1")

In [ ]:
# 토큰이 대화 기록 어디에도 노출되지 않았는지 확인
leaked = any(TOKEN in str(m.content) or TOKEN in str(getattr(m, "tool_calls", "")) for m in result["messages"])
print("🔒 토큰 노출 여부:", "노출됨 ❌" if leaked else "노출 없음 ✅")

### 💡 두 아키텍처의 심층 비교 (Trade-off Analysis)

| 비교 지표 | 방식 A: 정적 바인딩 (`MultiServerMCPClient`) | 방식 B: Skills 기반 점진적 공개 |
|:---|:---|:---|
| **프롬프트 토큰 소모** | ❌ **매우 큼**: 모든 서버의 모든 도구 스키마가 매 턴 주입 | ✅ **최소화**: 스킬 이름 1줄만 주입, 필요 시에만 문서 로드 |
| **도구 호출 지연 시간 (Latency)** | ✅ **빠름**: 도구 스키마를 이미 알고 있어 즉시 호출 | ⚠️ **다소 느림**: 매뉴얼 읽기 → 목록 확인 → 실행의 다단계 거침 |
| **도구 수 확장성 (Scalability)** | ❌ **낮음**: 도구가 20개 이상 넘어가면 도구 환각 발생 | ✅ **무제한**: 수백 개의 MCP 서버/도구도 카탈로그로 격리 가능 |
| **새 서버 추가/변경 시 작업** | ❌ **에이전트 코드 수정 및 전체 재배포 필수** | ✅ **`mcp_servers.json` 파일에 서버 정보 1개만 추가하면 즉시 인식** |
| **실무 적합 영역** | • 고정된 2~3개 도구만 사용하는 단일 목적 봇<br>• 밀리초 단위의 빠른 응답 속도가 최우선인 서비스 | • 사내 수십 개 마이크로서비스를 넘나드는 플랫폼 에이전트<br>• 도구 추가/삭제가 빈번한 엔터프라이즈 환경 |

> **아키텍처 인사이트:**
> Skills 방식은 초기 탐색 단계로 인해 호출 횟수와 지연 시간이 다소 늘어나지만, **"프롬프트 토큰 절감"**과 **"코드 수정 없는 무제한 확장성"**이라는 거대한 엔터프라이즈 이점을 제공합니다.

---
## 🏁 Part 6. 정리와 Mission 03

이번 시간에는 고립된 RAG 도구를 엔터프라이즈 표준 규격인 **FastMCP 서버**로 격리 구축하고,
**정적 바인딩**의 한계(Tool Explosion)를 넘어 **Skills 패턴(점진적 공개)**을 통해 에이전트가 스스로 도구를 탐색하고 실행하는 아키텍처를 배웠습니다.

실습을 마치기 전, 노트북에서 백그라운드로 띄웠던 테스트용 MCP 서버 프로세스를 안전하게 종료합니다.

In [ ]:
# 노트북에서 띄운 MCP 서버 종료 (이미 떠 있던 서버는 그대로 둡니다)
if mcp_process is not None:
    mcp_process.terminate()
    mcp_process.wait(timeout=10)
    print("🛑 MCP 서버 종료")

### 🎯 Mission 03. 인증된 MCP 서버와 Skills 에이전트 연동

`missions/03_missions.md`의 미션을 수행하세요.

1. **[미션 3-1] MCP 서버 완성**: `app/mcp/enterprise_rag_server.py`에 Bearer 인증기(`StaticTokenVerifier`)와 4대 RAG 도구(@mcp.tool)를 구현하고 CLI로 직접 검증합니다.
2. **[미션 3-2] 서버 레지스트리 등록**: `skills/mcp/references/mcp_servers.json`에 방금 띄운 서버의 URL과 환경 변수 이름을 등록합니다.
3. **[미션 3-3] Chainlit UI에서 두 에이전트 대화 비교**:
   - `tool_rag_agent`: 1교시의 도구 직접 바인딩 에이전트
   - `skill_rag_agent`: 이번 시간에 배운 Skills + MCP 자율 탐색 에이전트
   - 두 에이전트에게 동일한 질문을 던지고, Chainlit 단계별 로그(Step)에서 도구 호출 패턴과 토큰 사용량의 차이를 직접 눈으로 확인합니다.